# ASMSA v2 — joint multi-molecule AAE training

This notebook demonstrates the key new capability of v2: training **one** AAE jointly on frames from multiple molecules with different feature-vector lengths. Each molecule has its own linear input/output adapter; the encoder trunk, decoder trunk, latent space, and discriminator are all shared.

For speed the notebook uses randomly-generated feature tensors (two 'molecules' with 20 and 35 features). To run on real trajectories, replace cells 2 and 3 with:

```python
import mdtraj as md
from asmsa_v2 import featurize_frames
from asmsa_v2.featurize import Molecule

traj = md.load('traj_a.xtc', top='mol_a.pdb')
mol  = Molecule(pdb='mol_a.pdb', n_atoms=traj.n_atoms, diheds=...)
X_a  = featurize_frames(traj.xyz, mol.get_model())
```

In [ ]:
import torch

from asmsa_v2 import (
    AAE, Trainer, TrainConfig, IsotropicNormalPrior,
    MultiMolDataset, multi_mol_collate, set_seed,
)
from asmsa_v2.viz.latent import scatter_latents
from torch.utils.data import DataLoader

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
# 'molecule A' — e.g. alanine dipeptide: 20 internal coordinates
D_a = 20
N_a = 5000
mu_a = torch.randn(D_a) * 0.5
X_a = (mu_a + 0.4 * torch.randn(N_a, D_a))
print('mol A:', X_a.shape)

In [ ]:
# 'molecule B' — e.g. chignolin: 35 internal coordinates (different D!)
D_b = 35
N_b = 3000
mu_b = torch.randn(D_b) * 0.5
X_b = (mu_b + 0.4 * torch.randn(N_b, D_b))
print('mol B:', X_b.shape)

In [ ]:
ds = MultiMolDataset({'molA': X_a, 'molB': X_b})
loader = DataLoader(
    ds,
    batch_sampler=ds.round_robin_sampler(batch_size=128),
    collate_fn=multi_mol_collate,
)
print('batches per epoch:', len(loader))
print('feature dims:', ds.feature_dims)

In [ ]:
model = AAE(
    feature_dims=ds.feature_dims,
    latent_dim=2,
    encoder_widths=(128, 64, 32),
    decoder_widths=(32, 64, 128),
    disc_widths=(64, 64),
)
print(sum(p.numel() for p in model.parameters()), 'trainable params')

In [ ]:
cfg = TrainConfig(epochs=30, batch_size=128,
                  lr_ae=2e-4, lr_d=1e-4, lr_g=2e-4, device=device)
trainer = Trainer(model, prior=IsotropicNormalPrior(2), config=cfg)
history = trainer.fit(loader)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
for k, v in history.items():
    ax.plot(v, label=k)
ax.set_xlabel('epoch'); ax.set_yscale('log'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Encode frames from both molecules and overlay in the shared latent space
latents = trainer.encode_all({'molA': X_a, 'molB': X_b})
scatter_latents(latents)
plt.title('shared latent space'); plt.show()

In [ ]:
# Round-trip reconstruction MSE per molecule
model.eval()
with torch.no_grad():
    for mol_id, X in [('molA', X_a), ('molB', X_b)]:
        X = X.to(trainer.device)
        x_hat, _ = model(X, mol_id)
        mse = torch.mean((x_hat - X) ** 2).item()
        print(f'{mol_id}: recon MSE = {mse:.4f}')